# Autoregressive Models 🔄

## Introduction

In Lesson 2 you built a linear regression model using a single lag feature (`lag_1`) to forecast PM2.5. It worked: the model beat the persistence baseline and learned the coefficient `beta_1 ≈ 0.95` — the fingerprint of mean reversion in the series.

But that approach had a design flaw: you **manually chose one lag**. What if the series has memory that reaches back three hours? Five? Seven? You'd need to guess which lags to include and then treat them as arbitrary regression features, with no principled way to choose.

> ❓ Is there a framework that **automatically diagnoses how many lags matter** in a time series, and then fits a model with the right structure?

### From Linear Regression to AR Models

Recall the Lesson 2 model:

$$y_t = \beta_0 + \beta_1 y_{t-1} + \epsilon_t$$

An **Autoregressive model of order `p`**, written AR(`p`), generalises this to multiple lags:

$$y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + ... + \phi_p y_{t-p} + \epsilon_t$$

where:
- `y_t` is the value at time `t`
- `c` is a constant (long-run mean contribution)
- `phi_1, phi_2, ..., phi_p` are the autoregressive coefficients
- `epsilon_t` is white noise (random error at each step)
- `p` is the **order** of the model — how many past values are used

> 💡 **What changed from Lesson 2?** The equation is mathematically the same: it's a linear combination of past values plus noise. What changed is the **statistical framework** around it: `statsmodels` uses maximum likelihood estimation instead of OLS, provides confidence intervals for each `phi_i`, and gives you AIC/BIC scores for model comparison. The predictions will look similar; the diagnostic toolkit is far richer.

### Why AR Models?

Five reasons the AR framework improves on hand-rolled lag regression:

1. **Statistical rigour** — built on solid time series theory with known sampling properties
2. **Parameter efficiency** — the PACF plot tells you which lags carry signal; you don't blindly add columns to X
3. **Model selection criteria** — AIC and BIC compare models with different `p` values, accounting for the fit-complexity trade-off
4. **Coefficient confidence intervals** — `phi_i` estimates come with standard errors, not just point estimates
5. **Forecast machinery** — `model.forecast(steps=n)` handles the recursive prediction correctly; you don't need to write the loop yourself

### The Order Selection Problem

> 🚦 **The central question of this lesson:** given the PM2.5 time series, how many lags should we include?

- **Too few** (small `p`): underfitting — the model misses temporal patterns that exist in the data
- **Too many** (large `p`): overfitting — the model fits noise in the training set and predicts badly on new data
- **Just right**: we need a systematic tool to read off the right `p` from the data itself

That tool is the pair of diagnostic plots — ACF and PACF — which you will interpret before fitting any model.

## Learning Objectives

By the end of this lesson, you will be able to:

- Understand autoregressive (AR) models conceptually and relate them to Lesson 2's lag regression
- Interpret AR(`p`) models as regressions on lagged values with principled order selection
- Use ACF and PACF plots to determine the model order `p`
- Fit an AR model using `statsmodels.AutoReg` and interpret its coefficients
- Evaluate an AR model using walk-forward validation
- Compare AR models of different orders using MAE, AIC, and BIC

## Prepare Data

### Import

The `wrangle_data()` function from Lesson 1 gives us clean PM2.5 data in one call. We'll filter to 2024 data only for a manageable, recent window.

**Code Task 3.3.1.1**: Import the necessary libraries and load the PM2.5 data using `wrangle_data`. Get the data for 2024 only. Display the first few rows and the data shape.


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from load_mongo_data import load_nairobi_to_mongodb
from mongo_wrangle import wrangle_data

host = ...

# Connect to MongoDB on localhost
load_nairobi_to_mongodb(host=host)

# Load data
db = ...
collection = ...
df = wrangle_data(db=..., collection=..., host=host).loc['...']

print(f"Data shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print("\nFirst 5 rows:")
df.head()

## ACF and PACF — the intuition before the formulas

Before we plot anything, let's anchor *what* ACF and PACF measure with two concrete pictures.

**ACF (autocorrelation function) — "echo strength."** Imagine clapping in a canyon. The original clap is `y_t`. A faint echo bouncing back one second later is `y_{t-1}`'s influence on `y_t`. A fainter echo two seconds later is `y_{t-2}`'s influence. The **ACF at lag k** asks: *how strong is the echo at distance k?* For air quality, a PM2.5 spike at 8am leaves an echo at 9am (still elevated), a weaker echo at 10am, and so on. The ACF curve is the profile of those echoes across all lags.

**PACF (partial autocorrelation function) — "echo strength after removing intermediate echoes."** Now ask a sharper question: does *lag 3 specifically* add new information about today's PM2.5, *over and above* what lags 1 and 2 already tell us? That's PACF at lag 3. If lag 3's influence is entirely mediated through lags 1 and 2 — i.e. lag 1 → lag 2 → lag 3 → today is a chain — PACF at lag 3 will be near zero, even though ACF at lag 3 is sizeable.

**Why both?** For order selection: **PACF cuts off after lag *p*** in a pure AR(*p*) process, while **ACF tails off gradually**. The PACF cutoff is what we read off the plot to choose *p*. ACF confirms the autoregressive character. Together they are the time-series equivalent of using `.describe()` and a histogram side-by-side: one summarises, one diagnoses.

### Explore

With the canyon-echo intuition in mind, here is the **practical reading guide** for ACF/PACF plots:

> 🔍 **PACF cutoff rule (choose `p`):**
> 1. Look at the PACF plot. Find the **first lag after which all bars stay inside the blue confidence band**.
> 2. That lag number is your `p`.
> 3. Example: if PACF bars are outside the band at lags 1, 2, 3, 4, 5, but inside at lags 6, 7, ..., 20 → choose AR(5).
>
> **ACF tail-off rule (confirm AR character):**
> - In a pure AR(`p`) process, ACF decays **gradually and smoothly** toward zero rather than cutting off sharply.
> - A smoothly decaying ACF confirms the series has persistent memory — good news for AR modelling.
> - If ACF cuts off sharply (drops to near-zero after lag `q`) and PACF decays gradually, the series has MA character instead.

> ⚠️ **Parsimony principle:** when in doubt, choose the smaller `p`. An AR(3) that explains 95% of the predictable variance is almost always better in production than an AR(10) that explains 97% — the extra 2% is usually noise, and the extra 7 parameters cost you dearly in out-of-sample performance.


### Explore

Before fitting AR models, we need to understand the **autocorrelation
structure** of our data—how correlated are values with their past
selves?

**Code 3.3.1.1**: Create a function `plot_acf_pacf(series, lags=20)`
that plots both the Autocorrelation Function (ACF) and Partial
Autocorrelation Function (PACF) for the PM2.5 series. Use `statsmodels`
functions.

> **ACF vs PACF**
>
> - **ACF (Autocorrelation Function)**: Correlation between $y_t$ and
>   $y_{t-k}$ for all lags $k$
> - **PACF (Partial Autocorrelation Function)**: Correlation between
>   $y_t$ and $y_{t-k}$ after removing effects of intermediate lags
>
> These plots help us determine the AR order $p$: - **PACF cuts off
> after lag $p$** → suggests AR(p) model - **ACF tails off gradually** →
> confirms AR process

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def _acf_manual(x, nlags):
    """
    Compute the autocorrelation function up to nlags.
    
    Returns array of length nlags+1, with acf[0] = 1.0 by definition.
    Uses the biased estimator (denominator = n for all lags), which is
    the convention statsmodels uses by default.
    """
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    n = len(x)
    denom = np.dot(x, x)  # variance * n, equivalent to sum of squared deviations
    return np.array([np.dot(x[:n - k], x[k:]) / denom for k in range(nlags + 1)])


def _pacf_manual(x, nlags):
    """
    Compute the partial autocorrelation function up to nlags via Yule-Walker.
    
    For each lag k, fits an AR(k) model and takes the last coefficient as
    the PACF value at lag k. PACF[0] is 1.0 by definition.
    """
    acf_vals = _acf_manual(x, nlags)
    pacf_vals = [1.0]
    for k in range(1, nlags + 1):
        # Toeplitz matrix R of size k x k built from acf values
        R = np.array([[acf_vals[abs(i - j)] for j in range(k)] for i in range(k)])
        r = acf_vals[1:k + 1]
        phi = np.linalg.solve(R, r)
        pacf_vals.append(phi[-1])
    return np.array(pacf_vals)


def plot_acf_pacf(series, lags=20):
    """
    Plot ACF and PACF for a time series — manual implementation, no statsmodels.
    
    Parameters
    ----------
    series : pd.Series or array-like
        Time series data
    lags : int
        Number of lags to plot
    """
    values = np.asarray(series)
    n = len(values)
    
    acf_vals = _acf_manual(values, lags)
    pacf_vals = _pacf_manual(values, lags)
    
    # 95% confidence band: ±1.96/sqrt(n) is the standard approximation
    # (Bartlett's formula simplified for the null hypothesis of white noise)
    ci = 1.96 / np.sqrt(n)
    
    fig, axes = plt.subplots(1, 2, figsize=(9, 5))
    x_lags = np.arange(lags + 1)
    
    for ax, vals, title in [(axes[0], acf_vals, 'Autocorrelation Function (ACF)'),
                            (axes[1], pacf_vals, 'Partial Autocorrelation Function (PACF)')]:
        ax.vlines(x_lags, 0, vals, colors='steelblue', linewidth=1.5)
        ax.scatter(x_lags, vals, color='steelblue', zorder=3)
        ax.axhspan(-ci, ci, alpha=0.15, color='blue')  # 95% confidence band
        ax.axhline(0, color='black', linewidth=0.5)
        ax.set_title(title)
        ax.set_xlabel('Lag')
        ax.set_xticks(x_lags)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(-1.05, 1.05)
    
    plt.tight_layout()
    plt.show()


# Plot ACF and PACF for PM2.5 data
plot_acf_pacf(df['pm25'], lags=20)

> 📊 **Reading your ACF/PACF output:**
> - **PACF plot**: count the bars that extend beyond the blue shaded confidence band. The last lag that is clearly outside the band gives you a strong candidate for `p`. Bars that only barely peek outside the band should be treated with scepticism.
> - **ACF plot**: does it decay smoothly? A geometric-decay ACF (getting smaller each lag, never sharply cutting off) is the textbook signature of an AR process. If the ACF decays to near-zero within 2–3 lags, the series has limited memory and AR(1) or AR(2) may be sufficient.


> **Interpreting the Plots**
>
> Look at the PACF plot. After which lag do the bars drop into the
> confidence band (blue shaded area)? This indicates the order $p$ for
> your AR model!

**Code 3.3.1.2**: Based on the PACF plot, determine the appropriate AR
order $p$. Create a variable `ar_order` and assign your chosen value
(e.g., 3, 5, or 7). Print your reasoning.

> ⚠️ **Common mistake — reading PACF too aggressively.** PACF bars sit inside a blue confidence band. Bars *outside* the band are statistically significant; bars *inside* the band are noise. The mistake is to pick the largest lag whose bar pokes out anywhere on the plot — which gives you an AR(15) or AR(20) you do not need. The disciplined rule is: **find the first lag after which all bars stay inside the band**, and call that *p*. If PACF clearly cuts off after lag 5, an AR(5) is what the data is telling you to fit — even if lag 11 happens to poke out marginally. Trust the cutoff, not the cherry-pick.

In [ ]:
# Based on the PACF plot, determine the AR order
# Looking at typical air quality data, we'll use p=5 as an example
ar_order = 5

print(f"Chosen AR order: p = {ar_order}")
print(f"\nReasoning: The PACF plot shows significant spikes up to lag {ar_order},")
print(f"after which values drop into the confidence band (are not statistically significant).")
print(f"This suggests an AR({ar_order}) model is appropriate.")

### Split

**Code Task 3.3.1.2**: Split the data into training and testing sets
using an 80/20 chronological split. Create `train_data` and `test_data`
Series containing the PM2.5 values.

In [ ]:
# Calculate split index (80% for training)
split_idx = int(len(df) * 0.8)

# Split chronologically
train_data = df['pm25'].iloc[:...]
test_data = df['pm25'].iloc[...:]

print(f"Training set: {len(train_data)} observations")
print(f"Testing set: {len(test_data)} observations")
print(f"\nTraining period: {train_data.index.min()} to {train_data.index.max()}")
print(f"Testing period: {test_data.index.min()} to {test_data.index.max()}")

## Build Model

### Baseline

**Code Task 3.3.2.1**: Calculate the persistence baseline MAE for the
test set (predict that tomorrow equals today).

In [ ]:
from sklearn.metrics import mean_absolute_error

# Persistence baseline: predict using last observed value
y_pred_baseline = test_data.shift(...).dropna()
y_true_baseline = test_data[1:]  # Align with shifted predictions

# Calculate baseline MAE
mae_baseline = mean_absolute_error(y_true_baseline, ...)

print(f"Baseline (Persistence) MAE: {mae_baseline:.2f}")

### Iterate

> 💡 **`statsmodels.AutoReg` — fitting an AR model**
>
> ```python
> from statsmodels.tsa.ar_model import AutoReg
>
> model_ar = AutoReg(train_data, lags=ar_order)
> model_ar_fitted = model_ar.fit()
>
> print(model_ar_fitted.summary())
> ```
>
> The `AutoReg` class takes a **1D time series** (a pandas Series) and the number of lags `p`. It fits the AR(`p`) equation using ordinary least squares on the training data.
>
> **Key output attributes:**
>
> | Attribute | What it gives you |
> |-----------|-------------------|
> | `.params` | A Series with `const`, `pm25.L1`, ..., `pm25.L{p}` — the estimated coefficients |
> | `.aic` / `.bic` | AIC and BIC scores for model comparison |
> | `.forecast(steps=n)` | Generate `n` one-step-ahead forecasts from the fitted model |
> | `.conf_int()` | 95% confidence intervals for each coefficient |
>
> **`AutoReg` vs `ARIMA`:** for a pure AR(`p`) model (no MA component), `AutoReg` is simpler and faster. In Lesson 4 we'll use `ARIMA(p, 0, q)` when we add the MA component.


**Code Task 3.3.2.2**: Import `AutoReg` from `statsmodels` and fit an AR
model with order `ar_order` to the training data. Print the model
summary.

In [ ]:
from statsmodels.tsa.ar_model import AutoReg

# Fit AR model
model_ar = AutoReg(train_data, lags=...)
model_ar_fitted = model_ar.fit()

# Print summary
print(model_ar_fitted.summary())

> 📊 **Reading the `AutoReg` model summary:**
>
> When you call `model_ar_fitted.summary()`, statsmodels prints a detailed table. The key sections:
>
> - **`const` row**: the estimated intercept `c`. Check its p-value — if p > 0.05, the series may not have a significant long-run mean offset.
> - **`pm25.L1` through `pm25.L{p}` rows**: the estimated `phi_1` through `phi_p` coefficients. Look at both the estimate and the p-value. Coefficients with p > 0.10 are not statistically significant at the 10% level — their lag may not be worth including.
> - **Log-Likelihood, AIC, BIC**: these are at the bottom of the summary. Lower is better. You'll use these in Code 3.3.3.1 to compare orders.
> - **Durbin-Watson statistic**: tests for autocorrelation in the residuals. A value near 2.0 means the residuals are approximately uncorrelated — good. Values near 0 or 4 indicate remaining autocorrelation that the model hasn't captured.


> 🔮 **Predict before running:** Before we fit the AR(5), commit to a guess for the coefficient on `lag 1` (the most recent past). In Lesson 2 we found a single-lag coefficient close to 0.95 — and we interpreted the small distance below 1 as mean reversion. With five lags competing for explanatory power inside one model, the lag-1 coefficient will **split** with the longer lags. Will it stay close to 0.95? Drop to 0.5? Or even flip negative? Hold your guess. The point is that *adding lags to a model does not just add information — it redistributes the credit between the lags already there*.

> 💡 **Interpreting AR coefficients:**
>
> The `.params` Series from `model_ar_fitted` has entries named `const`, `pm25.L1`, `pm25.L2`, ..., `pm25.L{p}`.
>
> - **`const`** (`c`): the long-run mean contribution. When all `phi_i = 0`, the model predicts `const`. Equivalently, the long-run mean of the series is approximately `const / (1 - phi_1 - phi_2 - ... - phi_p)`.
> - **`pm25.L1`** (`phi_1`): the effect of one period ago on today, **after controlling for all other lags in the model**. This is the direct causal estimate, not the raw lag-1 correlation.
> - **`pm25.L2`** (`phi_2`): the additional effect of two periods ago, over and above lag 1's contribution.
>
> In a PM2.5 context, you often see `phi_1` near 0.8–0.95 (strong persistence from one hour ago) and `phi_2, ..., phi_p` much smaller. If some `phi_i` are **negative**, the model has detected oscillation — higher-than-average values at lag `i` slightly pull down today's value.


**Code Task 3.3.2.3**: Extract the AR coefficients and constant from the
fitted model. Create a DataFrame `coefficients_df` showing the lag
number and corresponding coefficient. Print the coefficients.

In [ ]:
# Extract coefficients
params = model_ar_fitted... # <-- pass in `params` here

# Handle different AutoReg versions
if 'const' in params.index:
    constant = params['const']
    ar_coeffs = params.drop('const')
else:
    constant = params.iloc[0] if len(params) > ar_order else 0
    ar_coeffs = params[-ar_order:] if len(params) > ar_order else params

# Create DataFrame
coefficients_df = pd.DataFrame({
    'Lag': range(1, len(ar_coeffs) + 1),
    'Coefficient': ar_coeffs.values
})

print("AR Model Coefficients:")
print(coefficients_df)
print(f"\nConstant: {constant:.4f}")

> 🤔 **Stop for a second.** Look at the AR(5) coefficients printed above. Are they all positive, or are some negative? An all-positive set says *"recent past values reinforce today"* — a simple decay-of-memory pattern. A *mix* of positive and negative coefficients says something more interesting: the model has detected **oscillation** (a higher-than-average value three hours ago slightly *pushes down* today's prediction). Which pattern does your fitted model show — and does it match what you would expect for hourly air-quality data?

## Walk-forward validation — what it actually simulates

A single chronological train/test split asks: *"train on Jan–Oct, freeze the model, predict Nov–Dec all at once."* That is honest but it is **not what a production forecaster does**. In production, you fit your model on whatever data you have *today*, predict *tomorrow*, then tomorrow you incorporate yesterday's true value and re-fit before predicting the day after. That *expanding-window, one-step-ahead* pattern is **walk-forward validation**.

The everyday analogy: imagine an exam where, after each question, you are told your answer, and you study that answer before the next question. The exam gets harder if questions drift in difficulty over time — but the way you study *also* improves as you get more questions under your belt. Walk-forward validation tests a model the way that exam tests a student: under the same information-acquisition pattern it will face in real life.

The cost is computational. Where a single split fits the model once and predicts many times, walk-forward fits the model *for every prediction*. For 30 predictions and a fast model like AR(5), that's negligible. For 30,000 predictions and a deep neural network, it is not. The principle is right; the practice gets prioritised.

> ✅ **The walk-forward loop pattern:**
>
> ```python
> predictions = []
> history = train_data.copy()   # start with all training data
>
> for t in range(len(test_data)):
>     model = AutoReg(history, lags=ar_order).fit()     # fit on history up to now
>     y_pred = model.forecast(steps=1)                   # predict ONE step ahead
>     predictions.append(y_pred.iloc[0])                 # store the prediction
>     history = pd.concat([history, test_data.iloc[[t]]])  # add the TRUE value to history
>
> y_pred_ar = pd.Series(predictions, index=test_data.index)
> ```
>
> Each iteration: fit the model → predict one step → append the actual (not the predicted) value to history. The window **expands** by one true observation at each step. This is why it's called "expanding window" validation.
>
> > ⚠️ **Computational cost:** this loop fits the model as many times as there are test observations. For AR(5) and 1,750 test points, that's 1,750 model fits. For speed in this lesson, we limit to `max_pred=30` predictions. The principle is identical; the computational cost would scale linearly with the test set size.


> 💡 **Transfer — walk-forward beyond time series.** The walk-forward idea shows up wherever a model must adapt as new evidence arrives. A chess engine updates its evaluation after every move played in a game. A streaming recommender re-ranks results after every click. An adaptive A/B test re-allocates traffic after every batch of conversions. In each case the *evaluation* of model quality has to match the *deployment* mode — measuring a model under static-batch conditions when production uses live updates is how teams get nasty surprises six months after launch.

**Code Task 3.3.2.4**: Use walk-forward validation to generate forecasts
for the test set. Start with the training data, make one-step-ahead
predictions, then expand the window. Store predictions in `y_pred_ar`.

> **Walk-Forward Validation**
>
> In time series, we simulate real forecasting by: 1. Starting with
> training data 2. Making a one-step-ahead prediction 3. Adding the
> actual observed value to our data 4. Re-fitting the model 5.
> Predicting the next step
>
> This is computationally expensive but provides the most realistic
> evaluation.

In [ ]:
# Initialize predictions list
predictions = []
history = ...copy() # <-- pass `train_data` here

# Walk-forward validation
for t in range(len(test_data)):
    # Fit model on current history
    model = AutoReg(history, lags=ar_order)
    model_fitted = model.fit()

    # Make one-step-ahead prediction
    y_pred = model_fitted.forecast(steps=1)
    predictions.append(y_pred.iloc[0])  # <-- use .iloc[0] to access Series value

    # Add actual observation to history
    history = pd.concat([history, test_data.iloc[[t]]])

# Convert predictions to Series
y_pred_ar = pd.Series(predictions, index=test_data.index)

print(f"Generated {len(y_pred_ar)} walk-forward predictions")
print(f"\nFirst 5 predictions:")
y_pred_ar.head()

> 🎯 **What does "beating the baseline" mean here?**
>
> The persistence baseline MAE (calculated in Code Task 3.3.2.1) is the bar we must clear. If `mae_ar < mae_baseline`, AR adds value. If they are nearly equal, the AR model is doing roughly what "tomorrow = today" does — not worth the complexity.
>
> A useful framing: express the improvement as a percentage — `(mae_baseline - mae_ar) / mae_baseline * 100`. A 5–10% improvement on real-world PM2.5 data is meaningful; a <2% improvement on a 30-sample walk-forward window is probably noise.


### Evaluate

**Code Task 3.3.2.5**: Calculate MAE, RMSE, and R² for the AR model on
the test set (limited to the number of predictions made). Compare with
the baseline.

> **⚠️ A Caution on R² for Time Series Models**
>
> In traditional regression, R² measures the proportion of variance
> explained by the model. For autocorrelated time series data, R² can be
> **misleading**.
>
> **Why?** R² compares predictions against the global mean of the entire
> series. But walk-forward validation doesn’t predict from the mean—it
> predicts from the most recent observation (the persistence baseline).
>
> An AR model can: - Consistently predict closer to the truth (lower MAE
> ✅) - But explain very little variance (low R² ❌)
>
> **Example:** If PM2.5 ranges 10-100 μg/m³ and your AR model improves
> MAE from 15 to 12 μg/m³, that’s good! But R² might still be 0.15
> because the model doesn’t capture all the ups and downs (variance).
> This doesn’t mean the model is bad—it means R² isn’t the right metric
> for time series.
>
> **Use MAE and RMSE instead** to evaluate time series forecasts.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Align test data with predictions
y_true_ar = test_data.iloc[:len(y_pred_ar)]

# Calculate metrics
mae_ar = mean_absolute_error(y_true_ar, ...)
rmse_ar = np.sqrt(mean_squared_error(y_true_ar, ...))
r2_ar = r2_score(y_true_ar, ...)

# Create comparison DataFrame (note: R² included for reference only)
comparison_df = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'AR Model': [mae_ar, rmse_ar, r2_ar],
    'Baseline': [mae_baseline, np.nan, np.nan]
})

print("Model Performance Comparison:")
print(comparison_df.round(4))
print("\n⚠️  Note: Focus on MAE/RMSE for time series, not R²")

print(f"\n📊 Improvement over baseline:")
print(f"   Baseline MAE: {mae_baseline:.2f}")
print(f"   AR Model MAE: {mae_ar:.2f}")
if mae_ar < mae_baseline:
    improvement = ((mae_baseline - mae_ar) / mae_baseline) * 100
    print(f"   ✅ {improvement:.1f}% better than baseline")
else:
    print(f"   ⚠️ Model did not beat baseline")

**Code 3.3.2.1**: Create a time series plot showing actual vs AR model
predictions for the test period. Also create a bar plot of the AR
coefficients.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6))

# Plot 1: Actual vs Predicted
axes[0].plot(y_true_ar.index, y_true_ar, label='Actual', alpha=0.7, linewidth=2)
axes[0].plot(y_pred_ar.index, y_pred_ar, label='AR Predictions', alpha=0.7, linewidth=2)
axes[0].set_xlabel('Time')
axes[0].set_ylabel('PM2.5 Concentration')
axes[0].set_title(f'AR({ar_order}) Model: Actual vs Predicted\n(Predictions appear "smoothed" due to mean reversion)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: AR Coefficients
axes[1].bar(coefficients_df['Lag'], coefficients_df['Coefficient'])
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('Coefficient Value')
axes[1].set_title(f'AR({ar_order}) Coefficients')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

> 📊 **Interpreting the AR model output plots:**
> - **Actual vs predicted (time series):** does the AR model track the actual PM2.5 better than the persistence baseline? The persistence model always predicts exactly one step behind. The AR model may anticipate changes slightly better — especially when the series is mean-reverting.
> - **AR coefficient bar chart:** are the bars decreasing in magnitude from lag 1 to lag `p`? Decreasing magnitudes confirm the "recency bias" of the AR model — recent history dominates, older history fades. Any bar that is significantly negative (and statistically significant) indicates oscillatory dynamics at that lag distance.


> **Why Do AR Predictions Look "Smoothed"?**
>
> Notice that AR predictions are less extreme than the actual data. When PM2.5 spikes to 80, the model predicts ~76. When it drops to 20, the model predicts ~23. This is **mean reversion in action**.
>
> AR models learn: "Extreme values don't persist. Next period is probably closer to the average." This is actually *correct behaviour* — it's why the model beats the baseline even though it can't perfectly capture sudden spikes.
>
> **This doesn't mean the model is broken.** It means AR models trade perfect spike prediction for consistent, reliable forecasts that revert toward realistic levels. For most decision-making applications (e.g., "should we issue a pollution advisory for tomorrow?"), better average accuracy over time matters more than nailing individual spikes.

## Communicate Results

### Comparing Different AR Orders: Statistics vs Practicality

When selecting the AR order `p`, two tools can give you conflicting signals:

1. **PACF cutoff** — reads off `p` from the diagnostic plot
2. **Walk-forward MAE** — measures empirical performance on held-out data

They can and do disagree — especially on small test windows. Here is the resolution:

> 🚦 **When PACF and MAE disagree:**
> - If PACF says AR(5) and walk-forward MAE says AR(3) is 0.05 lower — trust PACF. A 0.05 MAE difference on 30 predictions is noise; PACF is based on the full training set.
> - If PACF says AR(5) but AR(1) has lower AIC/BIC — trust AIC/BIC over walk-forward MAE for small test windows.
> - If walk-forward MAE for AR(7) is 10% lower than AR(5) — investigate, because a 10% improvement on 30 predictions is unusual and might indicate the data has weekly seasonality.

The `evaluate_ar_order()` function in the next Code Task lets you compare all candidate orders systematically.

**Code 3.3.3.1**: Create a function `evaluate_ar_order(order)` that fits an AR model with the specified order and calculates both MAE and AIC/BIC. Test orders 1, 3, 5, and 7 to compare.

> **⚠️ Important Context: Limited Test Data**
>
> Our walk-forward validation only uses **30 observations** from a test set of ~1,750. An MAE difference of 0.06 between AR(5) and AR(7) on a 30-sample window is **noise, not signal**. The right move when MAE differences this small disagree with PACF or AIC/BIC is to trust the statistical criteria, not the empirical number.


In [ ]:
from statsmodels.tsa.ar_model import AutoReg

def evaluate_ar_order(order, train_data, test_data, max_pred=30):
    """
    Evaluate AR model: walk-forward validation MAE + AIC/BIC.
    """
    predictions = []
    history = train_data.copy()

    for t in range(min(max_pred, len(test_data))):
        # Fit model
        model = AutoReg(history, lags=order)
        model_fitted = model.fit()

        # Predict and store AIC/BIC from first fit
        y_pred = model_fitted.forecast(steps=1)
        predictions.append(y_pred.iloc[0])  # <-- use .iloc[0] for Series

        # Update history
        history = pd.concat([history, test_data.iloc[[t]]])

    # Calculate MAE
    y_true = test_data.iloc[:len(predictions)]
    mae = mean_absolute_error(y_true, predictions)

    # Get AIC/BIC from model fit on full training data
    full_model = AutoReg(train_data, lags=order).fit()
    aic = full_model.aic
    bic = full_model.bic

    return mae, aic, bic

# Test different orders
orders = [1, 3, 5, 7]
results_dict = {}

for order in orders:
    print(f"Testing AR({order})...")
    mae, aic, bic = evaluate_ar_order(order, train_data, test_data, max_pred=30)
    results_dict[order] = {'MAE': mae, 'AIC': aic, 'BIC': bic}
    print(f"  MAE: {mae:.2f} | AIC: {aic:.1f} | BIC: {bic:.1f}")

# Create comparison DataFrame
comparison_table = pd.DataFrame(results_dict).T
print("\n" + "="*70)
print("MODEL COMPARISON:")
print(comparison_table.round(2))
print("="*70)

print(f"\n📊 Interpretation:")
print(f"   PACF suggests AR(5) - look where PACF plot cuts off")
print(f"   MAE shows: AR(7) slightly better ({results_dict[7]['MAE']:.2f}) vs AR(5) ({results_dict[5]['MAE']:.2f})")
print(f"   Difference: {abs(results_dict[7]['MAE'] - results_dict[5]['MAE']):.2f} (likely NOISE with n=30 test samples!)")
print(f"\n✅ CONCLUSION: Choose AR(5) based on:")
print(f"   - PACF statistical evidence (parsimony principle)")
print(f"   - AIC/BIC prefer AR(5) (penalize complexity)")
print(f"   - MAE difference (0.06) too small to be reliable")
print(f"   → AR(5) avoids overfitting while maintaining performance")

> 📊 **Reading the order comparison table:**
> - **Lowest AIC** — the model that best balances fit quality and number of parameters (with a moderate complexity penalty). Prefer this when predictive performance is the goal.
> - **Lowest BIC** — stricter penalty on complexity than AIC. Prefer this when parsimony is the goal (BIC will tend to prefer smaller `p`).
> - **Lowest MAE (walk-forward)** — the empirically best model on the 30-prediction window. Treat with caution: a 30-sample window is small, and rank differences below ~5% should be treated as ties.
> - **Verdict**: when AIC/BIC and MAE agree, the choice is clear. When they disagree, trust AIC/BIC on small windows and walk-forward MAE on large ones (hundreds of predictions).


> ⚠️ **Common mistake — chasing 0.06 MAE differences as if they were meaningful.** Walk-forward validation in this lesson uses only ~30 predictions out of a test set of ~1,750. An MAE difference of 0.06 between AR(5) and AR(7) on a 30-sample window is **noise, not signal**. The right move when MAE differences this small disagree with PACF or AIC/BIC is to trust the statistical criteria, not the empirical number. The mistake is seductive because lower MAE *feels* like evidence — but evidence needs sample size, and 30 is not enough. As a rule of thumb: when MAE differences between candidate models are smaller than ~10% of the baseline MAE, treat them as ties and break the tie with model simplicity (AIC/BIC), not with empirical rank.

## Summary

This lesson introduced **autoregressive (AR) models** — the time-series specialist's version of what Lesson 2 did with linear regression on a single lag.

### What You Built

| Step | Tool | Key insight |
|------|------|-------------|
| Diagnosed the series | ACF + PACF plots | PACF cutoff → choose `p` |
| Fit the AR model | `statsmodels.AutoReg(lags=p)` | Max-likelihood estimation, not OLS |
| Interpreted coefficients | `.params` attribute | Each `phi_i` is the *direct* effect of lag `i` |
| Evaluated fairly | Walk-forward validation | Simulates production: re-fit → predict → update |
| Compared orders | `evaluate_ar_order()` | AIC/BIC + MAE, not just MAE |

### Key Insights

- **PACF is the order-selection oracle.** For a pure AR(`p`) process, PACF cuts off sharply after lag `p`. Reading the PACF cutoff is the principled way to choose `p` — not guessing, not grid-searching every order.
- **Walk-forward is the honest evaluation.** A single chronological split tests the model once. Walk-forward tests it the way it would actually be used in production: re-fit on expanding history, predict one step, incorporate the true observation, repeat. The cost is computation; the benefit is realism.
- **Mean reversion explains the "smoothed" predictions.** AR coefficients near-but-below 1 systematically pull extreme predictions toward the series mean. That pull is not a modelling failure — it is the model correctly reflecting that extreme values don't persist.
- **AIC/BIC + MAE tell different stories.** Information criteria penalise model complexity on the full training data; MAE measures performance on a small held-out window. Neither is universally better. Use both, and be sceptical of MAE differences smaller than ~5% on short windows.

### What the AR Framework Still Can't Do

AR models capture **persistence** (today's value depends on past values). But they have no mechanism for capturing **shocks** — sudden PM2.5 spikes caused by a traffic jam, an industrial incident, or a dust storm, which elevate the series for a few hours and then dissipate.

> ❗️ **But wait:** if yesterday's model *missed* a pollution spike by +20 µg/m³, the AR model's next prediction uses the observed 8am value (80 µg/m³) — not the model's error. The model only sees values, not its own mistakes. That means past forecast errors carry no direct information into future predictions. The **Moving Average (MA) component** in Lesson 4 fills exactly this gap.

➡️ In Lesson 4, we'll add the MA component to get ARMA(`p`, `q`) — a model that uses both past values *and* past forecast errors. We'll also use AIC and BIC formally to select the best (`p`, `q`) combination and build the final model comparison table across all four modelling approaches in this project.

### Limitations and Next Steps

The AR(`p`) model is a significant upgrade over Lesson 2's single-lag linear regression. But it still has a blind spot:

- **AR only sees past values, not past errors.** When the model makes a large forecast error (say, it misses a pollution spike by 20 µg/m³), that information disappears. The next prediction doesn't know "the model was surprised yesterday; adjust accordingly."
- **Seasonal patterns require large `p`.** If PM2.5 has a weekly cycle (higher on weekdays than weekends), capturing it with an AR model requires lags up to `p = 168` (hours in a week). That's a lot of parameters.
- **Shocks fade, but AR can't model the fading independently.** A dust storm adds a one-off +30 µg/m³ that dissipates over 3 hours. AR models this as a persistent shift fed forward through the lag coefficients — a roundabout way to model what the MA component captures directly.

**You can take the Multiple Choice Questions**

## Reflection Questions

Before moving on, consider these questions:

1. Look at the AR coefficients your model printed. Do they decay monotonically toward zero, or do they oscillate (some positive, some negative)? What does each pattern tell you about how PM2.5 propagates hour-to-hour?

2. Compare a single chronological train/test split with walk-forward validation. Under what conditions would the two give meaningfully different estimates of the same model's performance — and which one would you trust for deciding whether to deploy?

3. The AR(`p`) model captures how past values influence the present. But what about past *forecast errors* — the times the model was surprised? Can you think of a scenario in PM2.5 monitoring where knowing "the model was off by +20 µg/m³ yesterday" would improve today's prediction, above and beyond what the raw lag-1 value provides?
